# Compartment 279: satellite-only research screening
Private Kaggle/Colab hosted CPU. Internet On; Accelerator None.
Three 2025 seasonal windows; 20 m analysis grid; bounded crops.
Rank up to 12 quality crops inside the polygon per season; process at most 3 candidates.
WorldCover 2021 crop is a historical weak reference, not verified forest labels.
Exports calibrated bands, numerical features, masks and a verified ZIP. No labels or trained model.
Failed seasonal coverage stays missing; it is never replaced by predictions.


In [ ]:
STUDY = None


In [ ]:
"""Fresh Phase 0 experiment: bounded real-data inspection in a hosted runtime."""
import hashlib
import importlib.metadata
import json
import math
import platform
import shutil
import time
import zipfile
from datetime import datetime, timezone
from pathlib import Path
from urllib.error import URLError
from urllib.request import urlopen

COLLECTION = 'sentinel-2-c1-l2a'
SCENE = 'S2C_T43QFE_20250329T053314_L2A'
BBOX = [76.785, 22.413, 76.805, 22.433]
BANDS = [('blue', 'B02'), ('green', 'B03'), ('red', 'B04'), ('nir', 'B08')]


def run(bbox=None, area_label='provisional research crop; not the Joga beat boundary', scene_id=SCENE):
    if platform.system() == 'Windows':
        raise RuntimeError('Use a hosted Kaggle or Colab CPU runtime; local processing is disabled.')
    if Path('/kaggle/working').is_dir():
        base, provider = Path('/kaggle/working'), 'Kaggle'
    elif Path('/content').is_dir():
        try:
            import google.colab
        except ImportError as error:
            raise RuntimeError('Use a hosted Colab runtime.') from error
        base, provider = Path('/content'), 'Colab'
    else:
        raise RuntimeError('Use a hosted Kaggle or Colab CPU runtime; local processing is disabled.')

    bbox = BBOX if bbox is None else list(bbox)
    if (len(bbox) != 4 or not all(math.isfinite(v) for v in bbox)
            or not -180 <= bbox[0] < bbox[2] <= 180
            or not -90 <= bbox[1] < bbox[3] <= 90):
        raise ValueError('Provide west/south/east/north bounds in longitude/latitude degrees.')

    packages = {}
    for name in ['numpy', 'rasterio', 'Pillow']:
        try:
            packages[name] = importlib.metadata.version(name)
        except importlib.metadata.PackageNotFoundError as error:
            raise RuntimeError(f'Cloud package missing: {name}. No automatic installation performed.') from error
    import numpy as np
    import rasterio
    from PIL import Image
    from rasterio.vrt import WarpedVRT
    from rasterio.warp import Resampling, transform as project_points, transform_bounds
    from rasterio.windows import Window, from_bounds

    if shutil.disk_usage(base).free < 100 * 1024**2:
        raise RuntimeError('Cloud runtime needs at least 100 MiB free for this bounded experiment.')
    output = base / 'forestguard_phase0' / datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
    output.mkdir(parents=True)
    started = time.monotonic()
    if not isinstance(scene_id, str) or not scene_id.startswith('S2') or not all(c.isalnum() or c == '_' for c in scene_id):
        raise ValueError('Provide a Sentinel-2 scene identifier.')
    url = f'https://earth-search.aws.element84.com/v1/collections/{COLLECTION}/items/{scene_id}'
    try:
        with urlopen(url, timeout=30) as response:
            content = response.read(512001)
    except URLError as error:
        raise RuntimeError('Metadata request failed. Check cloud Internet and any account verification requirement.') from error
    if len(content) > 512000:
        raise ValueError('Metadata exceeds the 500 KiB safety limit.')
    item = json.loads(content)
    assert item['id'] == scene_id
    (output / 'source.json').write_bytes(content)
    assets = item['assets']
    calibration, measurements, masks = {}, [], []
    with rasterio.Env(GDAL_DISABLE_READDIR_ON_OPEN='EMPTY_DIR', GDAL_HTTP_TIMEOUT='60',
                      CPL_VSIL_CURL_ALLOWED_EXTENSIONS='.tif', GDAL_CACHEMAX=64*1024**2):
        with rasterio.open(assets['red']['href']) as reference:
            bounds = transform_bounds('EPSG:4326', reference.crs, *bbox, densify_pts=21)
            crop = from_bounds(*bounds, transform=reference.transform)
            left, top = math.floor(crop.col_off), math.floor(crop.row_off)
            right, bottom = math.ceil(crop.col_off+crop.width), math.ceil(crop.row_off+crop.height)
            assert 0 <= left < right <= reference.width and 0 <= top < bottom <= reference.height
            assert max(right-left, bottom-top) <= 512, 'Crop exceeds the research size limit'
            window = Window(left, top, right-left, bottom-top)
            crs, grid = reference.crs, reference.window_transform(window)
            full_grid = reference.transform
            shape = (bottom-top, right-left)
        for key, name in BANDS:
            asset = assets[key]
            assert asset['href'].startswith('https://e84-earth-search-sentinel-data.s3.us-west-2.amazonaws.com/')
            scale = asset['raster:bands'][0]['scale']
            offset = asset['raster:bands'][0]['offset']
            assert math.isfinite(scale) and scale > 0 and math.isfinite(offset)
            with rasterio.open(asset['href']) as band:
                assert band.crs == crs and band.transform == full_grid and band.res == (10., 10.)
                assert math.isclose(band.scales[0],scale) and math.isclose(band.offsets[0],offset), 'Calibration metadata/header mismatch'
                raw = band.read(1, window=window, masked=True)
            calibration[name] = {'scale': scale, 'offset': offset, 'href': asset['href']}
            measurements.append(raw.filled(0).astype('float32') * scale + offset)
            masks.append(~np.ma.getmaskarray(raw))
        with rasterio.open(assets['scl']['href']) as quality:
            with WarpedVRT(quality, crs=crs, transform=grid, height=shape[0], width=shape[1],
                           resampling=Resampling.nearest) as aligned:
                scl = aligned.read(1)

    values = np.stack(measurements)
    row, col = np.indices(shape)
    x, y = grid.c+(col+.5)*grid.a, grid.f+(row+.5)*grid.e
    lon, lat = project_points(crs, 'EPSG:4326', x.ravel(), y.ravel())
    lon, lat = np.asarray(lon).reshape(shape), np.asarray(lat).reshape(shape)
    study = (lon >= bbox[0]) & (lon <= bbox[2]) & (lat >= bbox[1]) & (lat <= bbox[3])
    valid = study & np.logical_and.reduce(masks) & np.isfinite(values).all(axis=0) & np.isin(scl, [4, 5, 6])
    assert valid.any(), 'No usable pixels: inspect another observation or research crop'
    values[:, ~valid] = -9999
    profile = dict(driver='GTiff', crs=crs, transform=grid, height=shape[0], width=shape[1], compress='deflate')
    with rasterio.open(output/'reflectance.tif', 'w', **profile, count=4, dtype='float32', nodata=-9999) as dst:
        dst.write(values)
        dst.descriptions = tuple(name for _, name in BANDS)
    for name, array in [('scl', scl), ('study', study.astype('uint8')), ('usable', valid.astype('uint8'))]:
        with rasterio.open(output/f'{name}.tif', 'w', **profile, count=1, dtype='uint8') as dst:
            dst.write(array, 1)
    rgb = (np.clip(values[[2, 1, 0]] / .3, 0, 1)*255).astype('uint8').transpose(1, 2, 0)
    Image.fromarray(np.dstack([rgb, valid.astype('uint8')*255])).save(output/'preview.png')

    # Reopen outputs: successful writes alone are insufficient evidence.
    for name, expected in [('reflectance', values), ('scl', scl), ('study', study), ('usable', valid)]:
        with rasterio.open(output/f'{name}.tif') as saved:
            assert saved.crs == crs and saved.transform == grid and saved.shape == shape
            assert saved.res == (10., 10.)
            actual = saved.read() if expected.ndim == 3 else saved.read(1)
            assert np.array_equal(actual, expected), f'Saved pixels differ: {name}'
    classes, counts = np.unique(scl[study], return_counts=True)
    report = {
        'status': 'real research sample; not forest labels or beat reporting',
        'scene_id': scene_id, 'collection':COLLECTION, 'acquisition': item['properties']['datetime'], 'metadata_url': url,
        'bbox_lon_lat': bbox, 'area_label':area_label, 'official_boundary_verified': False, 'shape': list(shape),
        'crs': str(crs), 'transform': list(grid)[:6], 'resolution_m': 10,
        'band_order': [name for _, name in BANDS], 'calibration': calibration,
        'reflectance_formula': 'raw * asset scale + asset offset, applied once',
        'calibration_limit': 'Collection 1 asset metadata matches all four raster headers; independent absolute radiometric validation not performed.',
        'quality_rule': 'all bands valid and finite; SCL 4/5/6; inside research-box pixel centers',
        'scl_resampling': 'nearest neighbor; native 20 m quality remains 20 m information',
        'study_pixels': int(study.sum()), 'usable_pixels': int(valid.sum()),
        'usable_fraction': float(valid.sum()/study.sum()),
        'scl_counts_inside_study': {str(int(k)): int(v) for k, v in zip(classes, counts)},
        'reflectance_quantiles': {name: np.quantile(values[i, valid], [0, .01, .5, .99, 1]).tolist()
                                 for i, (_, name) in enumerate(BANDS)},
        'runtime': {'provider': provider, 'python': platform.python_version(), 'packages': packages,
                    'gdal': rasterio.__gdal_version__},
        'processing_seconds': round(time.monotonic()-started, 3),
        'generated_at_utc': datetime.now(timezone.utc).isoformat(),
        'attribution': f'Contains modified Copernicus Sentinel data {item["properties"]["datetime"][:4]}',
        'license_url': 'https://cds.climate.copernicus.eu/licences/ec-sentinel',
        'model_trained': False, 'reviewed_labels_exist': False,
    }
    (output/'report.json').write_text(json.dumps(report, indent=2), encoding='utf-8')
    manifest = {}
    for file in sorted(output.iterdir()):
        with file.open('rb') as stream:
            digest = hashlib.file_digest(stream, 'sha256').hexdigest()
        manifest[file.name] = {'sha256': digest, 'bytes': file.stat().st_size}
    (output/'checksums.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')
    bundle = output/'forestguard_phase0.zip'
    with zipfile.ZipFile(bundle, 'w', zipfile.ZIP_DEFLATED) as archive:
        for name in [*manifest, 'checksums.json']:
            archive.write(output/name, arcname=name)
    with zipfile.ZipFile(bundle) as archive:
        assert archive.testzip() is None
        for name, record in manifest.items():
            assert hashlib.sha256(archive.read(name)).hexdigest() == record['sha256']
    print(json.dumps(report, indent=2))
    print('Verified sample bundle:', bundle, '\nBundle bytes:', bundle.stat().st_size)
    return output




In [ ]:
"""Numerical research features; these functions do not generate land-cover labels."""
import numpy as np


def research_features(values, band_order, usable):
    values = np.asarray(values, dtype='float32')
    usable = np.asarray(usable, dtype=bool)
    if values.ndim != 3 or values.shape[1:] != usable.shape or values.shape[0] != len(band_order):
        raise ValueError('Band stack and usable mask must share a grid.')
    if len(set(band_order)) != len(band_order):
        raise ValueError('Band names must be unique.')
    required = {'B02','B04','B05','B08','B8A','B11'}
    if not required <= set(band_order):
        raise ValueError('Required visible/NIR/red-edge/SWIR bands are missing.')
    band = dict(zip(band_order, values))
    valid = usable & np.isfinite(values).all(axis=0)

    def ratio(numerator, denominator):
        result = np.full(usable.shape, np.nan, dtype='float32')
        np.divide(numerator, denominator, out=result, where=valid & (denominator > 1e-6))
        return result

    ndvi = ratio(band['B08']-band['B04'], band['B08']+band['B04'])
    ndre = ratio(band['B8A']-band['B05'], band['B8A']+band['B05'])
    ndmi = ratio(band['B8A']-band['B11'], band['B8A']+band['B11'])
    evi = ratio(2.5*(band['B08']-band['B04']), band['B08']+6*band['B04']-7.5*band['B02']+1)
    index_valid = valid & np.isfinite(ndvi) & (np.abs(ndvi) <= 1)
    for index in [ndre, ndmi]:
        index_valid &= np.isfinite(index) & (np.abs(index) <= 1)
    padded = np.pad(np.where(index_valid, ndvi, 0), 1)
    padded_valid = np.pad(index_valid, 1, constant_values=False)
    count = np.zeros(usable.shape, dtype='uint8')
    total = np.zeros(usable.shape, dtype='float32')
    squares = total.copy()
    height, width = usable.shape
    for dy in range(3):
        for dx in range(3):
            local = padded[dy:dy+height, dx:dx+width]
            count += padded_valid[dy:dy+height, dx:dx+width]
            total += local
            squares += local*local
    texture = np.sqrt(np.maximum(0, squares/9-(total/9)**2))
    features = np.concatenate([values, np.stack([ndvi,evi,ndre,ndmi,texture])])
    feature_valid = index_valid & (count == 9) & np.isfinite(features).all(axis=0)
    features[:, ~feature_valid] = np.nan
    return features, feature_valid, list(band_order)+['NDVI','EVI','NDRE_B8A_B05','NDMI_B8A_B11','NDVI_STD_3X3']


In [ ]:
"""Hosted-only bounded 279 research experiment; supplied helpers reuse the sample pipeline."""
import hashlib
import json
import math
import platform
import shutil
import zipfile
from datetime import datetime, timezone
from pathlib import Path
from urllib.request import Request, urlopen

EXTRA_BANDS = [('rededge1','B05'),('rededge2','B06'),('rededge3','B07'),
               ('nir08','B8A'),('swir16','B11'),('swir22','B12')]
SEASONS = [('dry','2025-02-01T00:00:00Z/2025-04-30T23:59:59Z'),
           ('wet','2025-07-01T00:00:00Z/2025-09-30T23:59:59Z'),
           ('post_monsoon','2025-10-01T00:00:00Z/2025-12-31T23:59:59Z')]
SCL_SCREEN_LIMIT = 12
PROCESS_LIMIT = 3


def scl_coverage(classes, inside):
    import numpy as np
    if classes.shape != inside.shape or not inside.any():
        raise ValueError('Quality crop must share a nonempty study mask.')
    return float((np.isin(classes,[4,5,6]) & inside).sum()/inside.sum())


def run_research(boundary, base_run, feature_builder, reference_builder=None):
    if platform.system() == 'Windows' or not (Path('/kaggle/working').is_dir() or Path('/content').is_dir()):
        raise RuntimeError('Use the private hosted Kaggle/Colab notebook; local acquisition is disabled.')
    if boundary is None or boundary.get('type') != 'FeatureCollection' or len(boundary.get('features',[])) != 1:
        raise ValueError('Supply the selected 279 study polygon in the private notebook.')
    feature = boundary['features'][0]
    if feature['properties'].get('study_id') != 'handia-compartment-279':
        raise ValueError('Research input must identify the selected compartment 279 study.')
    base = Path('/kaggle/working') if Path('/kaggle/working').is_dir() else Path('/content')
    if shutil.disk_usage(base).free < 250*1024**2:
        raise RuntimeError('At least 250 MiB of hosted free storage is required.')
    import numpy as np
    import rasterio
    from rasterio.features import geometry_mask
    from rasterio.vrt import WarpedVRT
    from rasterio.warp import Resampling, reproject, transform_bounds, transform_geom
    from rasterio.windows import Window, from_bounds
    from PIL import Image

    output = base/'forestguard_279_research'/datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
    output.mkdir(parents=True)
    (output/'boundary.geojson').write_text(json.dumps(boundary,indent=2))
    results, failures, screens = [], [], []
    bbox = boundary['bbox']
    for season, interval in SEASONS:
        query = {'collections':['sentinel-2-c1-l2a'],'bbox':bbox,'datetime':interval,'limit':30,
                 'sortby':[{'field':'properties.eo:cloud_cover','direction':'asc'}]}
        request = Request('https://earth-search.aws.element84.com/v1/search',
                          data=json.dumps(query).encode(),headers={'Content-Type':'application/json'})
        try:
            with urlopen(request,timeout=60) as response:
                catalog_bytes = response.read(2*1024**2+1)
            if len(catalog_bytes) > 2*1024**2:
                raise ValueError('Season catalogue exceeds 2 MiB.')
            catalog = json.loads(catalog_bytes)
            (output/f'{season}_catalogue.json').write_bytes(catalog_bytes)
            candidates = sorted(catalog.get('features',[]),
                key=lambda item:(item['properties'].get('eo:cloud_cover',100),item['id']))[:SCL_SCREEN_LIMIT]
        except Exception as error:
            failures.append({'season':season,'stage':'catalogue','error':str(error)})
            continue
        if not candidates:
            failures.append({'season':season,'stage':'catalogue','error':'No candidate acquisitions returned.'})
        ranked=[]
        for item in candidates:
            try:
                href=item['assets']['scl']['href']
                if not href.startswith('https://e84-earth-search-sentinel-data.s3.us-west-2.amazonaws.com/'):
                    raise ValueError('Unexpected quality asset host.')
                with rasterio.Env(GDAL_DISABLE_READDIR_ON_OPEN='EMPTY_DIR',GDAL_HTTP_TIMEOUT='60',GDAL_CACHEMAX=32*1024**2):
                    with rasterio.open(href) as quality:
                        bounds=transform_bounds('EPSG:4326',quality.crs,*bbox,densify_pts=21)
                        raw_window=from_bounds(*bounds,transform=quality.transform)
                        left,top=math.floor(raw_window.col_off),math.floor(raw_window.row_off)
                        right,bottom=math.ceil(raw_window.col_off+raw_window.width),math.ceil(raw_window.row_off+raw_window.height)
                        if not (quality.res==(20.,20.) and 0<=left<right<=quality.width and 0<=top<bottom<=quality.height
                                and max(right-left,bottom-top)<=256):
                            raise ValueError('Invalid/oversized native quality window.')
                        window=Window(left,top,right-left,bottom-top)
                        classes=quality.read(1,window=window,masked=True).filled(0)
                        inside=geometry_mask([transform_geom('EPSG:4326',quality.crs,feature['geometry'])],
                                             classes.shape,quality.window_transform(window),invert=True,all_touched=False)
                fraction=scl_coverage(classes,inside)
                screens.append({'season':season,'scene_id':item['id'],'acquisition':item['properties']['datetime'],
                                'tile_cloud_percent':item['properties'].get('eo:cloud_cover'),
                                'local_scl_coverage_fraction':fraction,'inside_study_pixels':int(inside.sum())})
                ranked.append((fraction,item))
            except Exception as error:
                failures.append({'season':season,'scene_id':item['id'],'stage':'quality_screen','error':str(error)})
        ranked.sort(key=lambda pair:(-pair[0],pair[1]['id']))
        for _,item in ranked[:PROCESS_LIMIT]:
            try:
                cropped = base_run(bbox=bbox, area_label='Compartment 279 selected research polygon', scene_id=item['id'])
                source = json.loads((cropped/'source.json').read_text())
                assets = source['assets']
                with rasterio.Env(GDAL_DISABLE_READDIR_ON_OPEN='EMPTY_DIR', GDAL_HTTP_TIMEOUT='60', GDAL_CACHEMAX=64*1024**2):
                    href = assets['rededge1']['href']
                    if not href.startswith('https://e84-earth-search-sentinel-data.s3.us-west-2.amazonaws.com/'):
                        raise ValueError('Unexpected asset host.')
                    with rasterio.open(href) as reference:
                        bounds = transform_bounds('EPSG:4326',reference.crs,*bbox,densify_pts=21)
                        window = from_bounds(*bounds,transform=reference.transform)
                        left,top = math.floor(window.col_off),math.floor(window.row_off)
                        right,bottom = math.ceil(window.col_off+window.width),math.ceil(window.row_off+window.height)
                        if not (0<=left<right<=reference.width and 0<=top<bottom<=reference.height and max(right-left,bottom-top)<=256):
                            raise ValueError('Invalid or oversized native-20 m crop.')
                        window = Window(left,top,right-left,bottom-top)
                        grid,crs,shape,full_grid = reference.window_transform(window),reference.crs,(bottom-top,right-left),reference.transform
                        if reference.res != (20.,20.):
                            raise ValueError('Red-edge reference must have native 20 m resolution.')
                    if results and (list(shape)!=results[0]['shape'] or str(crs)!=results[0]['crs'] or list(grid)[:6]!=results[0]['transform']):
                        raise ValueError('Seasonal observations must share the first accepted grid.')
                    with rasterio.open(cropped/'reflectance.tif') as saved:
                        with WarpedVRT(saved,crs=crs,transform=grid,height=shape[0],width=shape[1],resampling=Resampling.average) as aligned:
                            visible = aligned.read(masked=True)
                    support = np.zeros(shape,dtype='float32')
                    with rasterio.open(cropped/'usable.tif') as saved:
                        reproject(saved.read(1),support,src_transform=saved.transform,src_crs=saved.crs,
                                  dst_transform=grid,dst_crs=crs,resampling=Resampling.average)
                    values = list(visible.filled(np.nan))
                    band_order = ['B02','B03','B04','B08']
                    calibration = json.loads((cropped/'report.json').read_text())['calibration']
                    for key,name in EXTRA_BANDS:
                        asset = assets[key]
                        if not asset['href'].startswith('https://e84-earth-search-sentinel-data.s3.us-west-2.amazonaws.com/'):
                            raise ValueError('Unexpected asset host.')
                        scale,offset = asset['raster:bands'][0]['scale'],asset['raster:bands'][0]['offset']
                        with rasterio.open(asset['href']) as band:
                            if band.crs != crs or band.transform != full_grid or band.res != (20.,20.):
                                raise ValueError('Native 20 m bands do not share the reference grid.')
                            if not (math.isfinite(scale) and scale>0 and math.isfinite(offset) and
                                    math.isclose(band.scales[0],scale) and math.isclose(band.offsets[0],offset)):
                                raise ValueError('Calibration/header mismatch.')
                            raw = band.read(1,window=window,masked=True)
                        values.append(raw.filled(0).astype('float32')*scale+offset)
                        values[-1][np.ma.getmaskarray(raw)] = np.nan
                        band_order.append(name)
                        calibration[name] = {'scale':scale,'offset':offset,'href':asset['href']}
                inside = geometry_mask([transform_geom('EPSG:4326',crs,feature['geometry'])],shape,grid,invert=True,all_touched=False)
                if not inside.any():
                    raise ValueError('The study polygon has no raster pixel centres.')
                stack = np.stack(values)
                usable = inside & (support>=.999999) & np.isfinite(stack).all(axis=0) & ~np.ma.getmaskarray(visible).any(axis=0)
                features,feature_valid,names = feature_builder(stack,band_order,usable)
                fraction = float(feature_valid.sum()/inside.sum())
                if fraction < .9:
                    failures.append({'season':season,'scene_id':source['id'],'stage':'quality',
                                     'usable_fraction_inside_study':fraction,'error':'Below 90% feature-coverage screening target.'})
                    continue
                folder = output/season
                folder.mkdir()
                profile = dict(driver='GTiff',crs=crs,transform=grid,height=shape[0],width=shape[1],compress='deflate')
                for name,array,descriptions in [('reflectance',stack,band_order),('features',features,names)]:
                    stored = np.where(np.isfinite(array),array,-9999).astype('float32')
                    with rasterio.open(folder/f'{name}.tif','w',**profile,count=len(array),dtype='float32',nodata=-9999) as dst:
                        dst.write(stored); dst.descriptions = tuple(descriptions)
                    with rasterio.open(folder/f'{name}.tif') as saved:
                        if saved.transform != grid or saved.crs != crs or not np.array_equal(saved.read(),stored):
                            raise ValueError('Saved raster grid/pixels differ from computed outputs.')
                for name,array in [('study_mask',inside),('quality_mask',usable),('usable_mask',feature_valid)]:
                    with rasterio.open(folder/f'{name}.tif','w',**profile,count=1,dtype='uint8') as dst:
                        dst.write(array.astype('uint8'),1)
                    with rasterio.open(folder/f'{name}.tif') as saved:
                        if not np.array_equal(saved.read(1),array):
                            raise ValueError('Saved mask differs from computed mask.')
                rgb = (np.clip(np.nan_to_num(stack[[2,1,0]],nan=0)/.3,0,1)*255).astype('uint8').transpose(1,2,0)
                Image.fromarray(np.dstack([rgb,feature_valid.astype('uint8')*255])).save(folder/'preview.png')
                record = {'season':season,'scene_id':source['id'],'acquisition':source['properties']['datetime'],
                          'shape':list(shape),'crs':str(crs),'transform':list(grid)[:6],'analysis_resolution_m':20,
                          'band_order':band_order,'feature_order':names,'calibration':calibration,
                          'inside_study_pixels':int(inside.sum()),'usable_feature_pixels':int(feature_valid.sum()),
                          'usable_fraction_inside_study':fraction,'source_crop_path':str(cropped),
                          'source_crop_bundle':'source_crop.zip',
                          'quality_rule':'All included bands valid; SCL 4/5/6 support for all contributing 10 m pixels; valid 3x3 index neighborhood',
                          'resampling':'B02/B03/B04/B08 averaged from 10 m to native red-edge 20 m grid; extra bands read at native 20 m; no upsampling',
                          'texture_footprint_m':60,'model_trained':False,'labels_generated':False,
                          'source_license_url':'https://cds.climate.copernicus.eu/licences/ec-sentinel',
                          'attribution':f'Contains modified Copernicus Sentinel data {source["properties"]["datetime"][:4]}'}
                (folder/'source.json').write_text(json.dumps(source,indent=2))
                (folder/'report.json').write_text(json.dumps(record,indent=2))
                shutil.copy2(cropped/'forestguard_phase0.zip',folder/'source_crop.zip')
                results.append(record)
                print('Verified research crop:',season,source['id'],fraction)
                break
            except Exception as error:
                failures.append({'season':season,'scene_id':item['id'],'stage':'processing','error':str(error)})
                if (output/season).exists():
                    break  # Keep a partial failed write for diagnosis; never replace it with another scene.
    reference=None
    if reference_builder is not None:
        try:
            reference=reference_builder(output,boundary,results)
        except Exception as error:
            failures.append({'stage':'weak_reference','error':str(error)})
    report = {'study_id':'handia-compartment-279','status':'COMPLETE_DATA_SCREENING' if len(results)==len(SEASONS) else 'PARTIAL_DATA_SCREENING',
              'acquisitions':results,'failed_attempts':failures,'quality_screens':screens,'season_windows':SEASONS,
              'scl_screen_limit_per_season':SCL_SCREEN_LIMIT,'candidate_limit_per_season':PROCESS_LIMIT,'weak_reference':reference,
              'labels_reviewed':0,'model_trained':False,'independent_model_accuracy_measured':False,
              'cloud_runtime':{'python':platform.python_version(),'numpy':np.__version__,'rasterio':rasterio.__version__,'gdal':rasterio.__gdal_version__}}
    (output/'research_report.json').write_text(json.dumps(report,indent=2))
    manifest = {str(path.relative_to(output)):{'sha256':hashlib.sha256(path.read_bytes()).hexdigest(),'bytes':path.stat().st_size}
                for path in output.rglob('*') if path.is_file()}
    (output/'checksums.json').write_text(json.dumps(manifest,indent=2))
    bundle = output/'forestguard_279_research.zip'
    with zipfile.ZipFile(bundle,'w',zipfile.ZIP_DEFLATED) as archive:
        for name in [*manifest,'checksums.json']:
            archive.write(output/name,arcname=name)
    with zipfile.ZipFile(bundle) as archive:
        if archive.testzip() is not None:
            raise ValueError('Export archive is corrupt.')
        for name,record in manifest.items():
            if hashlib.sha256(archive.read(name)).hexdigest()!=record['sha256']:
                raise ValueError('Export archive hash mismatch.')
    print(json.dumps(report,indent=2)); print('Verified research ZIP:',bundle)
    return output


In [ ]:
"""Bounded public WorldCover crop: historical weak reference, never forest ground truth."""
import json
import math
from urllib.request import Request, urlopen

WORLD_COVER_CLASSES={10:'Tree cover',20:'Shrubland',30:'Grassland',40:'Cropland',50:'Built-up',
                     60:'Bare / sparse vegetation',70:'Snow and ice',80:'Permanent water',
                     90:'Herbaceous wetland',95:'Mangroves',100:'Moss and lichen'}


def worldcover_tile(bbox):
    west,south,east,north=bbox
    lon,lat=math.floor(west/3)*3,math.floor(south/3)*3
    if not (lon<=west<east<=lon+3 and lat<=south<north<=lat+3):
        raise ValueError('This bounded experiment needs a study within one WorldCover tile.')
    return f'{"N" if lat>=0 else "S"}{abs(lat):02d}{"E" if lon>=0 else "W"}{abs(lon):03d}'


def prepare_worldcover(output,boundary,observations):
    import platform
    from pathlib import Path
    if platform.system()=='Windows' or not (Path('/kaggle/working').is_dir() or Path('/content').is_dir()):
        raise RuntimeError('Use the private hosted notebook for reference acquisition.')
    import numpy as np
    import rasterio
    from rasterio.features import geometry_mask
    from rasterio.warp import Resampling,reproject,transform_geom
    from rasterio.windows import Window,from_bounds
    from affine import Affine
    bbox=boundary['bbox']
    tile=worldcover_tile(bbox)
    url=f'https://esa-worldcover.s3.eu-central-1.amazonaws.com/v200/2021/map/ESA_WorldCover_10m_2021_v200_{tile}_Map.tif'
    folder=output/'weak_reference';folder.mkdir()
    with urlopen(Request(url,method='HEAD'),timeout=30) as response:
        source_bytes=int(response.headers['Content-Length'])
        etag=response.headers.get('ETag')
        if source_bytes>1024**3:
            raise ValueError('Source metadata exceeds the expected 1 GiB tile bound; no full download attempted.')
    with rasterio.Env(GDAL_DISABLE_READDIR_ON_OPEN='EMPTY_DIR',GDAL_HTTP_TIMEOUT='60',GDAL_CACHEMAX=32*1024**2):
        with rasterio.open(url) as source:
            if str(source.crs)!='EPSG:4326' or source.count!=1 or not np.allclose(source.res,[1/12000,1/12000],rtol=1e-6):
                raise ValueError('Unexpected WorldCover source CRS/band count/resolution.')
            window=from_bounds(*bbox,transform=source.transform)
            left,top=math.floor(window.col_off),math.floor(window.row_off)
            right,bottom=math.ceil(window.col_off+window.width),math.ceil(window.row_off+window.height)
            if not (0<=left<right<=source.width and 0<=top<bottom<=source.height and max(right-left,bottom-top)<=512):
                raise ValueError('Reference window is invalid or exceeds 512 pixels per side.')
            window=Window(left,top,right-left,bottom-top)
            values=source.read(1,window=window,masked=True).filled(0)
            grid=source.window_transform(window)
            colors=source.colormap(1)
    if not set(np.unique(values))<=set(WORLD_COVER_CLASSES)|{0}:
        raise ValueError('Unrecognized WorldCover class codes.')
    inside=geometry_mask([boundary['features'][0]['geometry']],values.shape,grid,invert=True,all_touched=False)
    if not inside.any():raise ValueError('Reference has no study pixel centres.')
    profile=dict(driver='GTiff',crs='EPSG:4326',transform=grid,height=values.shape[0],width=values.shape[1],
                 count=1,dtype='uint8',nodata=0,compress='deflate')
    with rasterio.open(folder/'worldcover_2021_native.tif','w',**profile) as saved:
        saved.write(values,1);saved.write_colormap(1,colors)
        saved.set_band_description(1,'WorldCover 2021 v200 class code; weak historical reference')
    with rasterio.open(folder/'worldcover_2021_native.tif') as saved:
        if saved.transform!=grid or not np.array_equal(saved.read(1),values):
            raise ValueError('Saved native reference differs from source crop.')
    aligned=[]
    for observation in observations:
        shape=observation['shape'];transform=Affine(*observation['transform'])
        target=np.zeros(shape,dtype='uint8')
        reproject(values,target,src_transform=grid,src_crs='EPSG:4326',dst_transform=transform,
                  dst_crs=observation['crs'],src_nodata=0,dst_nodata=0,resampling=Resampling.nearest)
        mask=geometry_mask([transform_geom('EPSG:4326',observation['crs'],boundary['features'][0]['geometry'])],
                          shape,transform,invert=True,all_touched=False)
        target[~mask]=0
        with rasterio.open(output/observation['season']/'worldcover_2021_weak.tif','w',**{
            **profile,'crs':observation['crs'],'transform':transform,'height':shape[0],'width':shape[1]}) as saved:
            saved.write(target,1);saved.write_colormap(1,colors)
        with rasterio.open(output/observation['season']/'worldcover_2021_weak.tif') as saved:
            if saved.transform!=transform or not np.array_equal(saved.read(1),target):
                raise ValueError('Saved aligned weak reference differs from computed crop.')
        codes,counts=np.unique(target[mask],return_counts=True)
        aligned.append({'season':observation['season'],'grid_class_pixel_counts':dict(zip(map(str,codes),map(int,counts)))})
    codes,counts=np.unique(values[inside],return_counts=True)
    record={'dataset':'ESA WorldCover 2021 v200','reference_year':2021,'tile_id':tile,'source_url':url,
            'source_tile_bytes':source_bytes,'source_etag':etag,'full_tile_downloaded':False,
            'native_crs':'EPSG:4326','native_resolution_degrees':1/12000,'native_shape':list(values.shape),
            'native_transform':list(grid)[:6],'class_mapping':WORLD_COVER_CLASSES,
            'class_pixel_counts_inside_study':dict(zip(map(str,codes),map(int,counts))),
            'aligned_grids':aligned,'categorical_resampling':'nearest neighbour to observation pixel centres',
            'license':'CC BY 4.0','license_url':'https://creativecommons.org/licenses/by/4.0/',
            'attribution':'© ESA WorldCover project 2021 / Contains modified Copernicus Sentinel data (2021) processed by ESA WorldCover consortium',
            'citation':'Zanaga et al. (2022). ESA WorldCover 10 m 2021 v200. https://doi.org/10.5281/zenodo.7254221',
            'reference_kind':'weak_map','reference_independent':False,'reviewed_labels_exist':False,
            'limits':'2021 predicted classes; tree cover includes plantations and agricultural trees. No conversion to verified current forest labels or area.'}
    (folder/'reference_report.json').write_text(json.dumps(record,indent=2))
    print('Verified small WorldCover weak reference:',tile,values.shape)
    return record


In [ ]:
research_output = run_research(STUDY, run, research_features, prepare_worldcover)
